# Recovery and wellness

Owner-run, local-only view of daily wellness, sleep and HRV. Outputs may contain private health data; never commit or share them.

In [ ]:
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

from stridelog.db.models import DailyAnalytics, HrvNight, SleepNight
from stridelog.notebook_data import read_table

In [ ]:
daily = read_table(DailyAnalytics, ["local_date", "metrics"])
sleep = read_table(SleepNight, ["local_date", "duration_s"])
hrv = read_table(HrvNight, ["local_date", "last_night_avg_ms"])
daily_rows = [{"local_date": row["local_date"], **row["metrics"]} for row in daily]
if not daily_rows:
    raise ValueError("No daily analytics are available; run stridelog analyze first.")
wellness = pl.DataFrame(daily_rows).select(
    "local_date",
    "resting_hr_7d_avg_bpm",
    "steps",
    "stress_avg",
    "body_battery_high",
    "body_battery_low",
)
wellness

In [ ]:
sleep_frame = pl.DataFrame(
    sleep,
    schema={"local_date": pl.Date, "duration_s": pl.Float64},
).sort("local_date")
hrv_frame = pl.DataFrame(
    hrv,
    schema={"local_date": pl.Date, "last_night_avg_ms": pl.Float64},
).sort("local_date")
figure = make_subplots(
    rows=2, cols=1, shared_xaxes=True, subplot_titles=["Resting heart rate (bpm)", "Average stress"]
)
for row, column in enumerate(("resting_hr_7d_avg_bpm", "stress_avg"), start=1):
    figure.add_trace(
        go.Scatter(x=wellness["local_date"].to_list(), y=wellness[column].to_list(), name=column),
        row=row,
        col=1,
    )
figure

In [ ]:
sleep_figure = None
if sleep_frame.height:
    sleep_figure = go.Figure(
        go.Scatter(x=sleep_frame["local_date"].to_list(), y=sleep_frame["duration_s"].to_list())
    ).update_layout(title="Sleep duration (seconds)")
else:
    print("No sleep rows are available.")
sleep_figure

In [ ]:
hrv_figure = None
if hrv_frame.height:
    hrv_figure = go.Figure(
        go.Scatter(x=hrv_frame["local_date"].to_list(), y=hrv_frame["last_night_avg_ms"].to_list())
    ).update_layout(title="Last-night HRV (milliseconds)")
else:
    print("No HRV rows are available.")
hrv_figure